In [2]:
import serial
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pandas as pd
import numpy as np

class SerialPortReader:
    def __init__(self, port, baudrate=9600, timeout=1):
        """Initialize the serial port reader.
        
        Args:
            port (str): Serial port (e.g., 'COM3' on Windows, '/dev/ttyUSB0' on Linux)
            baudrate (int): Baud rate of the connection
            timeout (float): Read timeout in seconds
        """
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.ser = None
        self.data_buffer = []
        
    def connect(self):
        """Connect to the serial port."""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                timeout=self.timeout
            )
            print(f"Connected to {self.port} at {self.baudrate} baud")
            return True
        except Exception as e:
            print(f"Error connecting to serial port: {e}")
            return False
    
    def disconnect(self):
        """Disconnect from the serial port."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            print(f"Disconnected from {self.port}")
    
    def read_line(self):
        """Read a single line from the serial port."""
        if not self.ser or not self.ser.is_open:
            print("Serial port not open")
            return None
        
        try:
            line = self.ser.readline().decode('utf-8').strip()
            return line
        except Exception as e:
            print(f"Error reading from serial port: {e}")
            return None
    
    def read_data(self, num_points=100, delay=0.1):
        """Read data points from the serial port and store in buffer.
        
        Args:
            num_points (int): Number of data points to read
            delay (float): Delay between reads in seconds
        """
        self.data_buffer = []
        
        for i in range(num_points):
            line = self.read_line()
            if line:
                try:
                    # Try to convert to float if it's a number
                    value = float(line)
                    self.data_buffer.append(value)
                except ValueError:
                    # If not a number, store as string
                    self.data_buffer.append(line)
                
                # Print the current value
                print(f"Reading {i+1}/{num_points}: {line}")
            time.sleep(delay)
        
        return self.data_buffer
    
    def plot_numeric_data(self):
        """Plot the numeric data from the buffer."""
        numeric_data = []
        for item in self.data_buffer:
            if isinstance(item, (int, float)):
                numeric_data.append(item)
            else:
                try:
                    numeric_data.append(float(item))
                except:
                    pass
        
        if not numeric_data:
            print("No numeric data to plot")
            return
        
        plt.figure(figsize=(10, 6))
        plt.plot(numeric_data)
        plt.title('Serial Port Data')
        plt.xlabel('Sample Number')
        plt.ylabel('Value')
        plt.grid(True)
        plt.show()
    
    def stream_plot(self, max_points=100, update_interval=0.5, duration=30):
        """Live plot data from the serial port.
        
        Args:
            max_points (int): Maximum number of points to display
            update_interval (float): How often to update the plot in seconds
            duration (float): How long to run the stream in seconds
        """
        plt.figure(figsize=(10, 6))
        
        data = []
        start_time = time.time()
        
        while (time.time() - start_time) < duration:
            line = self.read_line()
            if line:
                try:
                    value = float(line)
                    data.append(value)
                    # Keep only the latest max_points
                    if len(data) > max_points:
                        data = data[-max_points:]
                    
                    # Update the plot
                    clear_output(wait=True)
                    plt.clf()
                    plt.plot(data)
                    plt.title('Real-time Serial Data')
                    plt.xlabel('Sample Number')
                    plt.ylabel('Value')
                    plt.grid(True)
                    plt.tight_layout()
                    plt.show()
                    
                except ValueError:
                    print(f"Non-numeric data: {line}")
            
            time.sleep(update_interval)

# Example usage
if __name__ == "__main__":
    # This code won't run when imported as a module in Jupyter
    pass

# Jupyter notebook usage example:
# 
# 1. Set up your serial port reader
# reader = SerialPortReader(port='COM3')  # Windows example
# # reader = SerialPortReader(port='/dev/ttyUSB0')  # Linux example
# 
# 2. Connect to the serial port
# reader.connect()
# 
# 3. Read some data
# data = reader.read_data(num_points=50, delay=0.1)
# 
# 4. Plot the data
# reader.plot_numeric_data()
# 
# 5. Or use the real-time streaming plot
# reader.stream_plot(max_points=100, update_interval=0.2, duration=60)
# 
# 6. Always disconnect when done
# reader.disconnect()

In [4]:

reader = SerialPortReader(port="/dev/tty.usbmodem101")
reader.connect()
reader.stream_plot(max_points=100, update_interval=0.2, duration=60)


Connected to /dev/tty.usbmodem101 at 9600 baud
Non-numeric data: 12,-7,-110,-4,-599
Non-numeric data: Uni:0.03,0.03,9.93,0.0087,-0.0134,-0.0081,-11.05,-0.50,-59.98
Non-numeric data: Raw:16,33,8303,7,-12,-7,-118,5,-606
Non-numeric data: Uni:0.02,0.04,9.93,0.0084,-0.0140,-0.0081,-11.81,0.53,-60.63
Non-numeric data: Raw:16,30,8303,8,-12,-7,-116,-3,-597
Non-numeric data: Uni:0.02,0.04,9.93,0.0089,-0.0136,-0.0078,-11.60,-0.35,-59.72
Non-numeric data: Raw:3,20,8301,6,-12,-7,-107,7,-599
Non-numeric data: Uni:0.00,0.02,9.93,0.0075,-0.0136,-0.0081,-10.76,0.73,-59.95
Non-numeric data: Raw:10,18,8293,8,-12,-7,-104,6,-598
Non-numeric data: Uni:0.01,0.02,9.92,0.0089,-0.0137,-0.0082,-10.46,0.67,-59.81
Non-numeric data: Raw:14,18,8305,8,-13,-8,-102,4,-602
Non-numeric data: Uni:0.02,0.02,9.94,0.0093,-0.0148,-0.0089,-10.20,0.47,-60.22
Non-numeric data: Raw:8,30,8303,9,-13,-7,-106,7,-604
Non-numeric data: Uni:0.01,0.04,9.93,0.0099,-0.0145,-0.0084,-10.61,0.73,-60.45
Non-numeric data: Raw:15,30,8303,8,-13

<Figure size 1000x600 with 0 Axes>

In [6]:
reader.disconnect()


Disconnected from /dev/tty.usbmodem101
